<b><font size="6"> Cars 4 You: Predicting Car Values with ML </font></b><br><br>

`Group 40`

Ana Macedo (20250405)<br>
Catarina Mendinhas (20250422)<br>
Lourenço Silva (20250453)<br>
Maria Fonseca (20250380)<br>

### <font color= '#0400ffff'>**Methodology** </font><a class="anchor" id='top'></a>

- [1. Abstract](#1)
- [2. Import Libraries](#2)
- [3. Metadata](#3)
- [4. Import Datasets](#4)
- [5. Data Cleaning](#5)
    - [5.1. Numeric Features Correction](#5_1)
    - [5.2. Categorical Features Correction](#5_2)
        - [5.2.1. Correct the values](#5_2_1)
        - [5.2.2. Check corrections](#5_2_2)
- [6. Data Partition](#6)
- [7. Feature Engineering & Preprocessing](#7)
    - [7.1. Outliers Treatment](#7_1)
    - [7.2. Feature Engineering ](#7_2)
    - [7.3. Encoding](#7_3)
    - [7.4. Scaling Features](#7_4)
    - [7.5. Missing Values Imputation](#7_5)
- [8. Feature Selection](#8)
    - [8.1. Remove paintQuality%](#8_1)
    - [8.2. Filter Methods](#8_2)
        - [8.2.1. Removing Constant Features](#8_2_1)
        - [8.2.2. Correlation between features - Redundant Features](#8_2_2)
        - [8.2.3. Correlation with the target - Relevant Features](#8_2_3)
    - [8.3. Wrapped Methods](#8_3)
        - [8.3.1. RFE](#8_3_1)
    - [8.4. Embedded Methods](#8_4)
        - [8.4.1. Lasso](#8_4_1)
    - [8.5. Feature Importance](#8_5)
    - [8. Comparison between Models](#8_6)
- [9. Model and Evaluation](#9)


<a class="anchor" id="1">

# **1. Abstract**

[Back to TOP](#TOP)
</a>

This section investigates the impact of different Data Preprocessing techniques on the performance of our models. Keeping the model architecture and hyperparameters fixed, we systematically evaluate how individual preprocessing decisions affect predictive performance on both training and validation.
Falar tambem que vamos analisar aqui as brands

<a class="anchor" id="2">

# **2. Import Libraries**

[Back to TOP](#TOP)
</a>

To embark on this project, it is essential to import the necessary libraries which play a pivotal role in efficient data management and predictive model development. 

This approach employed the use of NumPy and Pandas for the manipulation of data, and Plotly for the creation of interactive visualisations. Statistical tests were conducted using SciPy, while the preprocessing and modelling were dependent on scikit-learn, including tools for imputation, scaling, and a variety of machine learning algorithms. Performance evaluation was carried out using MAE and $R^2$ metrics.

In [40]:
# General Useful Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from math import ceil
import math

# Text Similarity - Categorical Features Correction
from difflib import SequenceMatcher

# Scaling and Encoding
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder, MinMaxScaler, StandardScaler, RobustScaler

# Missing Value Imputation
from sklearn.impute import KNNImputer, SimpleImputer

# Data Partition
from sklearn.model_selection import train_test_split

#filter methods
from sklearn.feature_selection import VarianceThreshold
from scipy.stats import spearmanr

# spearman 
from sklearn.feature_selection import SelectKBest, f_regression

# mutual information
from sklearn.feature_selection import mutual_info_classif

#wrapper methods
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.feature_selection import RFE

# embedded methods
from sklearn.linear_model import Lasso

from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

# Linear Models
from sklearn.linear_model import Ridge, Lasso, ElasticNet

#KNN
from sklearn.neighbors import KNeighborsRegressor

# Random Forest
from sklearn.ensemble import RandomForestRegressor

#Neural Network
from sklearn.neural_network import MLPRegressor

# DecisionTree
from sklearn.tree import DecisionTreeRegressor

# Ensemble
from sklearn.ensemble import BaggingRegressor, ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor

#Model evaluation
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error, mean_absolute_percentage_error
import statsmodels.api as sm

# Load Created Functions
import sys
sys.path.append("../")

from Source.visualizations import *
from Source.data_correction_preprocessing import *
from Source.feature_engineering import *
from Source.feature_selection import *
from Source.model_and_assessment import *

# Set random seed for reproducibility
np.random.seed(40111)

<a class="anchor" id="3">

# **3. Metadata**

[Back to TOP](#TOP)
</a>

Understanding the features is essential to interpret the data correctly and conduct the subsequent analysis and modeling steps. This section provides a brief description of each feature in the dataset, explaining its meaning and type.

**carID:** An attribute that contains an identifier for each car.

**Brand:** The car's main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**fuelType:** The type of fuel used by the car (Diesel, Petrol, Hybrid, Electric).

**mpg:** The car's consumption of fuel expressed in average miles per gallon.

**engineSize:** Size of the engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.

<a class="anchor" id="4">

# **4. Import Datasets**

[Back to TOP](#TOP)
</a>

Here, we are importing with pd.read_csv() the files to start our project and converting them into Pandas DataFrames. CarID was set to index since it's the unique identifier in all data sets for each individual car.

In [2]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')

train.set_index('carID', inplace = True)
test.set_index('carID', inplace = True)

<a class="anchor" id="5">

# **5. Data Cleaning**

[Back to TOP](#TOP)
</a>

In this section, the objective is to clean the dataset by correcting inconsistencies identified during the exploratory data analysis (EDA) phase, across both numerical and categorical features. This process is supported by a dedicated .py file, which contains a set of helper functions developed to streamline and standardize the data cleaning workflow.

Data cleaning is a critical step toward the project’s final objective of accurately predicting the target variable. Errors and inconsistencies in the data can directly affect how the model learns underlying patterns, ultimately impacting its ability to generalize and produce reliable predictions. 

<a class="anchor" id="5_1">

## **5.1** Numeric Features Correction
[Back to TOP](#TOP)
</a>

From this analysis, it was concluded that no electric vehicles are associated with negative tax values. Therefore, this hypothesis was discarded, and all negative values were corrected using a custom function implemented in the data_preprocessing.py file.

In [3]:
train = correct_metric_features(train)
test = correct_metric_features(test)

<a class="anchor" id="5_2">

## **5.2** Categorical Features Correction

[Back to TOP](#TOP)

<a class="anchor" id="5_2_1">

### **5.2.1** Correct the values

[Back to TOP](#TOP)

**Correct values with misspelled words for categorical**

In [4]:
# Correct 'Brand' column
train = clean_with_diff(train, 'Brand', threshold_short=0.6, threshold_long=0.6)
test = clean_with_diff(test, 'Brand', threshold_short=0.6, threshold_long=0.6)

# Correct 'transmission' column
train = clean_with_diff(train, 'transmission', threshold_short=0.7, threshold_long=0.7)
test = clean_with_diff(test, 'transmission', threshold_short=0.7, threshold_long=0.7)

# Correct 'model' column
train = clean_with_diff(train, 'model', threshold_short=0.70, threshold_long=0.90)
test = clean_with_diff(test, 'model', threshold_short=0.70, threshold_long=0.90)

# Correct 'fuelType' column
train = clean_with_diff(train, 'fuelType', threshold_short=0.8, threshold_long=0.8)
test = clean_with_diff(test, 'fuelType', threshold_short=0.8, threshold_long=0.8)

**Replace Other by Unknown**

In [5]:
train = replace_category_transmission(train)
test = replace_category_transmission(test)

<a class="anchor" id="6">

# **6. Data Partition**

[Back to TOP](#TOP)

In this section, the dataset is partitioned into independent and dependent variables, where the independent features are defined as X and the target variable (price) as y. This separation is essential to clearly distinguish the inputs used by the model from the value it is expected to predict.

Subsequently, the data is split into training and validation sets using the holdout method. This step allows the models to be trained on a subset of the data while being evaluated on unseen observations, providing a more realistic assessment of their generalization capability. This is also a critical measure to prevent data leakage, which occurs when information from the validation set influences the training process. 

After the split, all preprocessing steps that rely on statistical properties of the data (such as outlier detection, imputation, or scaling) are computed exclusively using the training set and then applied to the validation set.

This approach ensures that model evaluation remains unbiased and that the reported performance metrics accurately reflect how the model would behave when exposed to new, unseen data.

In [6]:
X = train.drop('price', axis = 1)
y = train['price']

X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.2, random_state = 42,  shuffle = True)

<a class="anchor" id="7">

# **7. Feature Engineering & Preprocessing**

[Back to TOP](#TOP)

<a class="anchor" id="7_1">

## **7.1** Original Data Preprocessing

[Back to TOP](#TOP)

In [ ]:
# Original Data Sets
X_train_original = X_train.copy()
y_train_original = y_train.copy()
X_val_original = X_val.copy()
y_val_original = y_val.copy()
test_original = test.copy()

In [ ]:
# Outliers Treatment
X_train_clean = treat_outliers_custom(X_train_original, X_train_original)
X_val_clean = treat_outliers_custom(X_train_original, X_val_original)
test_clean = treat_outliers_custom(X_train_original, test_original)

# Update Original Data Sets with cleaned versions
X_train_original = X_train_clean.copy()
X_val_original = X_val_clean.copy()
test_original = test_clean.copy()

In [45]:
# Feature Engineering
X_train_fe = create_features(X_train_original, X_train_original, current_year=2020, threshold=3)
X_val_fe = create_features(X_train_original, X_val_original, current_year=2020, threshold=3)
test_fe = create_features(X_train_original, test_original, current_year=2020, threshold=3)

# Update Original Data Sets to include new features
X_train_original = X_train_fe.copy()  
X_val_original = X_val_fe.copy()
test_original = test_fe.copy()

KeyError: 'fuelType'

In [ ]:
for brand in X_train_original['Brand'].unique():


In [ ]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_original = data_preprocessing(X_train_original, y_train_original, X_train_original, neighbors=5, imputation_method="knn", scaling_method="standard")
X_val_dp_original = data_preprocessing(X_train_original, y_train_original, X_val_original, neighbors=5, imputation_method="knn", scaling_method="standard")
test_dp_original = data_preprocessing(X_train_original, y_train_original, test_original, neighbors=5, imputation_method="knn", scaling_method="standard")

# Update Original Data Sets to include Preprocessing
X_train_original = X_train_dp_original.copy()
X_val_original = X_val_dp_original.copy()
test_original = test_dp_original.copy()

<a class="anchor" id="7_2">

## **7.2** Data Preprocessing Without Outlier Treatment

[Back to TOP](#TOP)

In [ ]:
# Data Sets to not treat outliers
X_train_with_outliers = X_train.copy()
y_train_with_outliers = y_train.copy()
X_val_with_outliers = X_val.copy()
y_val_with_outliers = y_val.copy()
test_with_outliers = test.copy()

In [ ]:
# Feature Engineering
X_train_fe_wo = create_features(X_train_with_outliers, X_train_with_outliers, current_year=2020, threshold=3)
X_val_fe_wo = create_features(X_train_with_outliers, X_val_with_outliers, current_year=2020, threshold=3)
test_fe_wo = create_features(X_train_with_outliers, test_with_outliers, current_year=2020, threshold=3)

# Update Data Sets to include new features
X_train_with_outliers = X_train_fe_wo.copy()  
X_val_with_outliers = X_val_fe_wo.copy()
test_with_outliers = test_fe_wo.copy()

In [ ]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_outliers = data_preprocessing(X_train_with_outliers, y_train_with_outliers, X_train_with_outliers, neighbors=5, imputation_method="knn", scaling_method="standard")
X_val_dp_outliers = data_preprocessing(X_train_with_outliers, y_train_with_outliers, X_val_with_outliers, neighbors=5, imputation_method="knn", scaling_method="standard")
test_dp_outliers = data_preprocessing(X_train_with_outliers, y_train_with_outliers, test_with_outliers, neighbors=5, imputation_method="knn", scaling_method="standard")

# Update Data Sets to include Preprocessing
X_train_with_outliers = X_train_dp_outliers.copy()
X_val_with_outliers = X_val_dp_outliers.copy()
test_with_outliers = test_dp_outliers.copy()

<a class="anchor" id="7_3">

## **7.3** Data Preprocessing With Different Scaling Methods

[Back to TOP](#TOP)

<a class="anchor" id="7_3_1">

### **7.3.1** MinMax Scaling

[Back to TOP](#TOP)

In [ ]:
# Data Sets to use Min-Max Scaling
X_train_minmax = X_train.copy()
y_train_minmax = y_train.copy()
X_val_minmax = X_val.copy()
y_val_minmax = y_val.copy()
test_minmax = test.copy()

In [ ]:
# Treat Outliers
X_train_clean_minmax = treat_outliers_custom(X_train_minmax, X_train_minmax)
X_val_clean_minmax = treat_outliers_custom(X_train_minmax, X_val_minmax)
test_clean_minmax = treat_outliers_custom(X_train_minmax, test_minmax)

# Update Data Sets with cleaned versions
X_train_minmax = X_train_clean_minmax.copy()
X_val_minmax = X_val_clean_minmax.copy()
test_minmax = test_clean_minmax.copy()

In [ ]:
# Feature Engineering
X_train_fe_minmax = create_features(X_train_minmax, X_train_minmax, current_year=2020, threshold=3)
X_val_fe_minmax = create_features(X_train_minmax, X_val_minmax, current_year=2020, threshold=3)
test_fe_minmax = create_features(X_train_minmax, test_minmax, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_minmax = X_train_fe_minmax.copy()  
X_val_minmax = X_val_fe_minmax.copy()
test_minmax = test_fe_minmax.copy()

In [ ]:
# Encoding, Scaling and Missing Value Imputation
X_train_dp_minmax = data_preprocessing(X_train_minmax, y_train_minmax, X_train_minmax, neighbors=5, imputation_method="knn", scaling_method="minmax")
X_val_dp_minmax = data_preprocessing(X_train_minmax, y_train_minmax, X_val_minmax, neighbors=5, imputation_method="knn", scaling_method="minmax")
test_dp_minmax = data_preprocessing(X_train_minmax, y_train_minmax, test_minmax, neighbors=5, imputation_method="knn", scaling_method="minmax")

# Update Data Sets to include Preprocessing
X_train_minmax = X_train_dp_minmax.copy()
X_val_minmax = X_val_dp_minmax.copy()
test_minmax = test_dp_minmax.copy()

<a class="anchor" id="7_3_2">

### **7.3.2** MinMax Scaling between -1 and 1

[Back to TOP](#TOP)

In [ ]:
# Data sets to use Min-Max Scaling between -1 and 1
X_train_minmax2 = X_train.copy()
y_train_minmax2 = y_train.copy()
X_val_minmax2 = X_val.copy()
y_val_minmax2 = y_val.copy()
test_minmax2 = test.copy()

In [19]:
# Data sets with outlier treatment
X_train_clean_minmax2 = treat_outliers_custom(X_train_minmax2, X_train_minmax2)
X_val_clean_minmax2 = treat_outliers_custom(X_train_minmax2, X_val_minmax2)
test_clean_minmax2 = treat_outliers_custom(X_train_minmax2, test_minmax2)

# Update original datasets with cleaned versions
X_train_minmax2 = X_train_clean_minmax2.copy()
X_val_minmax2 = X_val_clean_minmax2.copy()
test_minmax2 = test_clean_minmax2.copy()

In [20]:
X_train_fe_minmax2 = create_features(X_train_minmax2, X_train_minmax2, current_year=2020, threshold=3)
X_val_fe_minmax2 = create_features(X_train_minmax2, X_val_minmax2, current_year=2020, threshold=3)
test_fe_minmax2 = create_features(X_train_minmax2, test_minmax2, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_minmax2 = X_train_fe_minmax2.copy()  
X_val_minmax2 = X_val_fe_minmax2.copy()
test_minmax2 = test_fe_minmax2.copy()

In [21]:
X_train_dp_minmax2 = data_preprocessing(X_train_minmax2, y_train_minmax2, X_train_minmax2, neighbors=5, imputation_method="knn", scaling_method="minmax2")
X_val_dp_minmax2 = data_preprocessing(X_train_minmax2, y_train_minmax2, X_val_minmax2, neighbors=5, imputation_method="knn", scaling_method="minmax2")
test_dp_minmax2 = data_preprocessing(X_train_minmax2, y_train_minmax2, test_minmax2, neighbors=5, imputation_method="knn", scaling_method="minmax2")

X_train_minmax2 = X_train_dp_minmax2.copy()
X_val_minmax2 = X_val_dp_minmax2.copy()
test_minmax2 = test_dp_minmax2.copy()

<a class="anchor" id="7_3_3">

### **7.3.3** Robust Scaling

[Back to TOP](#TOP)

In [22]:
# Data sets without outlier treatment
X_train_robust = X_train.copy()
y_train_robust = y_train.copy()
X_val_robust = X_val.copy()
y_val_robust = y_val.copy()
test_robust = test.copy()

In [23]:
# Data sets with outlier treatment
X_train_clean_robust = treat_outliers_custom(X_train_robust, X_train_robust)
X_val_clean_robust = treat_outliers_custom(X_train_robust, X_val_robust)
test_clean_robust = treat_outliers_custom(X_train_robust, test_robust)

# Update original datasets with cleaned versions
X_train_robust = X_train_clean_robust.copy()
X_val_robust = X_val_clean_robust.copy()
test_robust = test_clean_robust.copy()

In [24]:
X_train_fe_robust = create_features(X_train_robust, X_train_robust, current_year=2020, threshold=3)
X_val_fe_robust = create_features(X_train_robust, X_val_robust, current_year=2020, threshold=3)
test_fe_robust = create_features(X_train_robust, test_robust, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_robust = X_train_fe_robust.copy()  
X_val_robust = X_val_fe_robust.copy()
test_robust = test_fe_robust.copy()

In [25]:
X_train_dp_robust = data_preprocessing(X_train_robust, y_train_robust, X_train_robust, neighbors=5, imputation_method="knn", scaling_method="robust")
X_val_dp_robust = data_preprocessing(X_train_robust, y_train_robust, X_val_robust, neighbors=5, imputation_method="knn", scaling_method="robust")
test_dp_robust = data_preprocessing(X_train_robust, y_train_robust, test_robust, neighbors=5, imputation_method="knn", scaling_method="robust")

X_train_robust = X_train_dp_robust.copy()
X_val_robust = X_val_dp_robust.copy()
test_robust = test_dp_robust.copy()

<a class="anchor" id="7_4">

## **7.4** Data Preprocessing With Different Imputation of Missing Values

[Back to TOP](#TOP)

In [26]:
# Data sets without outlier treatment
X_train_simple = X_train.copy()
y_train_simple = y_train.copy()
X_val_simple = X_val.copy()
y_val_simple = y_val.copy()
test_simple = test.copy()

In [27]:
# Data sets with outlier treatment
X_train_clean_simple = treat_outliers_custom(X_train_simple, X_train_simple)
X_val_clean_simple = treat_outliers_custom(X_train_simple, X_val_simple)
test_clean_simple = treat_outliers_custom(X_train_simple, test_simple)

# Update original datasets with cleaned versions
X_train_simple = X_train_clean_simple.copy()
X_val_simple = X_val_clean_simple.copy()
test_simple = test_clean_simple.copy()

In [28]:
X_train_fe_simple = create_features(X_train_simple, X_train_simple, current_year=2020, threshold=3)
X_val_fe_simple = create_features(X_train_simple, X_val_simple, current_year=2020, threshold=3)
test_fe_simple = create_features(X_train_simple, test_simple, current_year=2020, threshold=3)

# Update dataframes to include new features
X_train_simple = X_train_fe_simple.copy()  
X_val_simple = X_val_fe_simple.copy()
test_simple = test_fe_simple.copy()

In [30]:
X_train_dp_simple = data_preprocessing(X_train_simple, y_train_simple, X_train_simple, neighbors=5, imputation_method="simple", scaling_method="robust")
X_val_dp_simple = data_preprocessing(X_train_simple, y_train_simple, X_val_simple, neighbors=5, imputation_method="simple", scaling_method="robust")
test_dp_simple = data_preprocessing(X_train_simple, y_train_simple, test_simple, neighbors=5, imputation_method="simple", scaling_method="robust")

X_train_simple = X_train_dp_simple.copy()
X_val_simple = X_val_dp_simple.copy()
test_simple = test_dp_simple.copy()

<a class="anchor" id="1">

# **8. Feature Selection**

[Back to TOP](#TOP)
</a>


In [29]:
selected_features = ['year', 'mileage', 'mpg', 'engineSize', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 
                     'fuel_efficiency_score', 'tax_to_engine_ratio', 'brand_median_mileage', 'brand_avg_engineSize', 
                     'fueltype_avg_mpg', 'brand_avg_age', 'Brand_target', 'model_target', 'transmission_Semi-Auto_ohe']

In [46]:
# Feature Selection - Keep only selected features for all datasets
X_train_original = X_train_original[selected_features]
X_val_original = X_val_original[selected_features]

X_train_with_outliers = X_train_with_outliers[selected_features]
X_val_with_outliers = X_val_with_outliers[selected_features]

X_train_minmax = X_train_minmax[selected_features]
X_val_minmax = X_val_minmax[selected_features]

X_train_minmax2 = X_train_minmax2[selected_features]
X_val_minmax2 = X_val_minmax2[selected_features]

X_train_robust = X_train_robust[selected_features]
X_val_robust = X_val_robust[selected_features]

X_train_simple = X_train_simple[selected_features]
X_val_simple = X_val_simple[selected_features]

<a class="anchor" id="1">

# **9. Model and Evaluation**

[Back to TOP](#TOP)
</a>

<a class="anchor" id="5_1">

## **9.1** Create a Sample to Test Models

[Back to TOP](#TOP)
</a>

In this subsection, we create a sample of our datasets to test the modelling structure and ensure that the code is correctly implemented. Working with a smaller subset speeds up experimentation and allows us to verify the functionality of our models more efficiently. This approach also helps us identify potential issues earlier in the process.

In [47]:
X_train_s = X_train_original.sample(1000, random_state=42)
y_train_s = y_train_original.loc[X_train_s.index]
X_val_s = X_val_original.sample(500, random_state=42)
y_val_s = y_val_original.loc[X_val_s.index]

In [55]:
data_set_dict = {
    'Original': (X_train_original, y_train_original, X_val_original, y_val_original),
    'Without_Outlier_Treatment': (X_train_with_outliers, y_train_with_outliers, X_val_with_outliers, y_val_with_outliers),
    'MinMax': (X_train_minmax, y_train_minmax, X_val_minmax, y_val_minmax),
    'MinMax2': (X_train_minmax2, y_train_minmax2, X_val_minmax2, y_val_minmax2),
    'Robust': (X_train_robust, y_train_robust, X_val_robust, y_val_robust),
    'Simple_Imputation': (X_train_simple, y_train_simple, X_val_simple, y_val_simple)
}

In [56]:
estimator_dt = DecisionTreeRegressor(criterion = 'absolute_error', max_depth=18, min_samples_split=40, min_samples_leaf=10, max_features=0.7, ccp_alpha=0.001, random_state=42)
model_bag_dt = BaggingRegressor(estimator = estimator_dt, n_estimators=80, max_samples=0.9, max_features=0.8, bootstrap_features=False, bootstrap=True, random_state = 42, n_jobs=-1)

compare_model_dp(data_set_dict, model_bag_dt)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Robust,1326.794427,1419.231363,6.966937,0.929909,0.937348
Without_Outlier_Treatment,1328.089090,1419.596180,6.890132,0.928290,0.937367
Original,1334.653470,1428.639763,7.042000,0.929253,0.936174
MinMax,1339.038219,1432.893828,7.009181,0.928913,0.936303
MinMax2,1338.868752,1434.022773,7.107046,0.928946,0.935851
Simple_Imputation,1386.034505,1470.770597,6.113563,0.923860,0.931384


In [57]:
model_rf = RandomForestRegressor(n_estimators=100, min_samples_split=5, min_samples_leaf=5, max_samples=0.8, max_features='log2', max_depth=None, ccp_alpha=0.0, random_state=42)

compare_model_dp(data_set_dict, model_rf)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Without_Outlier_Treatment,1194.688659,1370.049994,14.678413,0.951871,0.945316
Robust,1193.946422,1372.546015,14.958761,0.952352,0.944706
Original,1193.924852,1374.863418,15.154938,0.952360,0.944208
MinMax,1201.353189,1378.868895,14.776313,0.951387,0.943650
MinMax2,1203.609057,1384.875857,15.060272,0.950819,0.943119
Simple_Imputation,1237.700584,1412.655852,14.135508,0.948601,0.939904


In [58]:
model_et = ExtraTreesRegressor(n_estimators=200, min_samples_split=8, min_samples_leaf=1, max_samples=0.8, max_features=None, max_depth=30, criterion='squared_error', bootstrap=True, random_state=42)

compare_model_dp(data_set_dict, model_et)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Without_Outlier_Treatment,962.464346,1300.542367,35.126290,0.970570,0.949855
Robust,962.998684,1306.877559,35.709174,0.970689,0.949308
Original,964.462428,1312.215310,36.056654,0.970998,0.949461
MinMax2,968.996629,1312.314119,35.430205,0.970655,0.948702
MinMax,969.988552,1313.592919,35.423549,0.970282,0.948678
Simple_Imputation,998.684788,1334.210742,33.596782,0.968887,0.945859


In [59]:
model_gb = GradientBoostingRegressor(subsample=0.75, n_estimators=1000, min_samples_split=10, min_samples_leaf=10, max_features=1.0, max_depth=5, loss='huber', learning_rate=0.04, random_state=42)

compare_model_dp(data_set_dict, model_gb)

,Train_MAE,Val_MAE,Gap_MAE_%,Train_R2,Val_R2
Data_Preprocessing,,,,,
Original,1261.630617,1350.353251,7.032378,0.951677,0.948499
Robust,1259.696020,1352.869074,7.396471,0.952381,0.948740
Without_Outlier_Treatment,1259.107897,1354.139813,7.547559,0.951613,0.948942
MinMax2,1267.034584,1361.792675,7.478730,0.951369,0.948138
MinMax,1268.100670,1366.048823,7.724005,0.950931,0.947755
Simple_Imputation,1284.530552,1375.665727,7.094823,0.950725,0.946059
